# Uruchomienie `app.py` (rozpoznawanie gestów dłoni)

Notatnik uruchamia demo z kamerki: `app.py` → okno **Hand Gesture Recognition**.

**Kolejność:** (1) sprawdzenie środowiska + pobranie modelu → (2) uruchomienie aplikacji.

`app.py` uruchamiamy jako podproces, żeby argparse nie kłócił się z argumentami jądra
Jupyter'a. Sterowanie działa w oknie OpenCV, nie w notatniku.

## 1. Środowisko i model

Sprawdza wersje bibliotek oraz pobiera `model/hand_landmarker.task`, jeśli pliku jeszcze nie ma.

Flaga `NO_CLASSIFICATION` (komórka poniżej) decyduje, czy `app.py` wystartuje **z**
klasyfikatorem gestów, czy **bez** niego (argument `--no_classification`).
Przy `True` plik `keypoint_classifier.tflite` nie jest wymagany.

`HEIGHT_DIFF_THRESHOLD` ustawia argument `--height_diff_threshold` (px):
jeśli ręce różnią się pionowo o więcej niż ten próg, klasyfikacja zwraca `inactive`.

`REST_ZONE_RADIUS` i `REST_ZONE_Y_OFFSET` (komórka uruchamiająca) ustawiają strefę spoczynku —
półokrąg przy dolnej krawędzi ekranu. Środek okręgu leży `REST_ZONE_Y_OFFSET` px **poniżej**
ekranu, promień to `REST_ZONE_RADIUS` px (widoczna wysokość = `r - y_offset`).
Gdy **obie** ręce są w strefie, gest jest klasyfikowany jako `inactive`. `REST_ZONE_RADIUS = 0` wyłącza strefę.

`GESTURE_COOLDOWN` (s) ustawia `--gesture_cooldown`: po zmianie rozpoznanego gestu kolejna zmiana
jest możliwa dopiero po tym czasie (domyślnie 0.6 s) — eliminuje migotanie różnych gestów.
W oknie widać wtedy `HOLD x.xs (next: ...)`. `0` wyłącza blokadę.

In [5]:
import sys
import urllib.request
from pathlib import Path
import cv2, mediapipe, numpy

ROOT = Path.cwd()
NO_CLASSIFICATION = False

# Model landmarkowania dłoni
MODEL_URL = ('https://storage.googleapis.com/mediapipe-models/hand_landmarker/'
             'hand_landmarker/float16/1/hand_landmarker.task')
model_file = ROOT / 'model' / 'hand_landmarker.task'

if not model_file.exists():
    print('Pobieram hand_landmarker.task')
    urllib.request.urlretrieve(MODEL_URL, model_file)

files = [model_file,
         ROOT / 'model/keypoint_classifier/keypoint_classifier_label.csv']
if not NO_CLASSIFICATION:  # klasyfikator potrzebny tylko z klasyfikacją
    files.append(ROOT / 'model/keypoint_classifier/keypoint_classifier.tflite')
for p in files:
    assert p.exists(), f'Brak pliku modelu: {p}'
if NO_CLASSIFICATION:
    print('TEST_CLASSIFICATION = True -> app.py wystartuje BEZ klasyfikatora')

## 2. Uruchomienie `app.py`

Zmień `DEVICE`, jeśli kamera nie jest na urządzeniu 0.
Komórka wypisuje uruchamiane polecenie — jeśli `TEST_CLASSIFICATION = True`,
dopisywany jest argument `--test_classification` (app bez klasyfikatora gestów).
Komórka blokuje się do zamknięcia okna aplikacji (**ESC** w oknie OpenCV).

In [6]:
import subprocess

DEVICE = 0

# Strefa spoczynku (półokrąg u dołu ekranu); 0 = wyłączona
REST_ZONE_RADIUS = 300    # promień r [px]
REST_ZONE_Y_OFFSET = 120   # środek okręgu tyle px PONIŻEJ dolnej krawędzi ekranu
GESTURE_COOLDOWN = 0.4


ARGS = [
    '--num_hands', '2',
    '--height_diff_threshold', '30',
    '--rest_zone_radius', str(REST_ZONE_RADIUS),
    '--rest_zone_y_offset', str(REST_ZONE_Y_OFFSET),
    '--gesture_cooldown', str(GESTURE_COOLDOWN),
    '--device', str(DEVICE),
    '--width', '960',
    '--height', '540',
]

if NO_CLASSIFICATION:
    ARGS.append('--no_classification')

print('app.py', ' '.join(ARGS))
proc = subprocess.run([sys.executable, 'app.py', *ARGS], cwd=ROOT)
print('Kod wyjścia:', proc.returncode)
if proc.returncode != 0:
    raise RuntimeError('app.py zakończył się błędem — patrz komunikaty powyżej.')

app.py --num_hands 2 --height_diff_threshold 30 --rest_zone_radius 300 --rest_zone_y_offset 120 --gesture_cooldown 0.4 --device 0 --width 960 --height 540


I0000 00:00:1794719422.898898  108914 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794719422.939528  108914 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794719424.059750  108914 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.4)


INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x7e2fc40036b0, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x7e2fc4071110, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
/home/krzysztof/Documents/hand-gesture-recognition-mediapipe-for-game-input/.CVenv/lib/python3.14/site-packages/tensorflow/lite/python/interpreter.py:468: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter fro

Keypoint classifier loaded: model/keypoint_classifier/keypoint_classifier.tflite
Gamepad connected: Xbox 360 Controller


W0000 00:00:1794719426.370077  108977 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
W0000 00:00:1794719426.370073  108976 inference_runner_litert.cc:1433] Input tensor memory is not aligned according to LiteRt's memory alignment requirements. Reallocating memory and copying data.
QFontDatabase: Cannot find font directory /home/krzysztof/Documents/hand-gesture-recognition-mediapipe-for-game-input/.CVenv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/krzysztof/Documents/hand-gesture-recognition-mediapipe-for-game-input/.CVenv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase

Kod wyjścia: 0


### Sterowanie (w oknie OpenCV)

| Klawisz / pad | Funkcja |
|---|---|
| `ESC` | wyjście z aplikacji |
| `n` | tryb normalny (bez zapisu danych) |
| `k` | tryb zapisu keypointów (do `model/keypoint_classifier/keypoint.csv`) |
| `1`–`5` | wybór kategorii (Push / Heal / Choke / Lightning / inactive) |
| D-pad góra/dół | wybór kategorii 1–5 (Push / Heal / Choke / Lightning / inactive) |
| LB lub RB | zapis jednej klatki wybranej kategorii |

> Zapis wymaga **obu rąk** w kadrze: jeden wiersz CSV = 2 grafy rąk = 84 cechy.
> Przy jednej ręce (lub żadnej) klasyfikacja pokazuje `inactive`.
> Po nagraniu danych wytrenuj model notatnikiem `keypoint_classification_EN.ipynb`.